## IMPORT CÁC THƯ VIỆN CẦN THIẾT

In [1]:
import pandas as pd
import numpy as np
import xgboost
from sklearn.model_selection import RandomizedSearchCV
import pickle

## TẢI DỮ LIỆU VÀ TIỀN XỬ LÝ

### Đọc dữ liệu

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
train = train.drop(train[(train['GrLivArea'] > 4000)].index)

### Xóa các cột thiếu dữ liệu quá nhiều và định danh Id

In [4]:
to_drop = ['Id', 'Alley', 'PoolQC', 'Fence', 'MiscFeature']
train.drop(columns=to_drop, inplace=True, errors='ignore')
test.drop(columns=to_drop, inplace=True, errors='ignore')

### Phân loại cột để điền giá trị khuyết

In [5]:
cat_col_train = ['FireplaceQu', 'GarageType', 'GarageFinish', 'MasVnrType', 'BsmtQual', 
                 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
                 'GarageQual', 'GarageCond']
ncat_col_train = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea']

cat_col_test = ['FireplaceQu', 'GarageType', 'GarageFinish', 'MasVnrType', 'BsmtQual', 
                'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
                'GarageQual', 'GarageCond', 'MSZoning', 'Utilities', 'Exterior1st', 
                'Exterior2nd', 'KitchenQual', 'Functional', 'SaleType']
ncat_col_test = ['LotFrontage', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 
                 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 
                 'GarageCars', 'GarageArea']

### Điền Mode cho biến phân loại và Mean cho biến liên tục

In [6]:
for i in cat_col_train:
    train[i] = train[i].fillna(train[i].mode()[0])
for j in ncat_col_train:
    train[j] = train[j].fillna(train[j].mean())

for i in cat_col_test:
    test[i] = test[i].fillna(test[i].mode()[0])
for j in ncat_col_test:
    test[j] = test[j].fillna(test[j].mean())


## KỸ THUẬT ĐẶC TRƯNG

### Tạo đặc trưng dựa trên kiến thức thực tế 

In [7]:
final_df = pd.concat([train, test], axis=0)

### Tính tuổi của ngôi nhà tại thời điểm bán

In [8]:
final_df['TotalSF'] = final_df['TotalBsmtSF'] + final_df['1stFlrSF'] + final_df['2ndFlrSF']
final_df['HouseAge'] = final_df['YrSold'] - final_df['YearBuilt']

cols_to_drop = ['TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'YearBuilt']
final_df.drop(columns=[c for c in cols_to_drop if c in final_df.columns], inplace=True)

all_cat_col = ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 
               'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 
               'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 
               'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 
               'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 
               'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional', 
               'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 
               'PavedDrive', 'SaleType', 'SaleCondition']

for field in all_cat_col:
    if field in final_df.columns:
        df_dummy = pd.get_dummies(final_df[field], drop_first=True, prefix=field)
        final_df.drop([field], axis=1, inplace=True)
        final_df = pd.concat([final_df, df_dummy], axis=1)

final_df = final_df.loc[:, ~final_df.columns.duplicated()]
num_train = len(train)
df_train = final_df.iloc[:num_train, :]
df_test = final_df.iloc[num_train:, :].drop(['SalePrice'], axis=1)

X_train = df_train.drop(['SalePrice'], axis=1)

### Biến đổi Logarit cho SalePrice

In [9]:
y_train = np.log1p(df_train['SalePrice'])

## HUẤN LUYỆN VÀ TỐI ƯU MÔ HÌNH

In [10]:
xgb_model = xgboost.XGBRegressor(random_state=42)

In [11]:
param = {
    'n_estimators': [100, 500, 900, 1100],
    'max_depth': [2, 3, 5, 10],
    'learning_rate': [0.05, 0.1, 0.15],
    'min_child_weight': [1, 2, 3],
    'booster': ['gbtree'],
    'base_score': [0.25, 0.5, 0.75]
}


In [12]:
random_cv = RandomizedSearchCV(estimator=xgb_model,
                               param_distributions=param,
                               cv=5, n_iter=20,
                               scoring='neg_mean_absolute_error', 
                               n_jobs=-1, verbose=1,
                               return_train_score=True,
                               random_state=42)

In [13]:
random_cv.fit(X_train, y_train)
best_xgb = random_cv.best_estimator_


Fitting 5 folds for each of 20 candidates, totalling 100 fits


## DỰ ĐOÁN VÀ XUẤT KẾT QUẢ

In [14]:
with open('xgb_model.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)

In [15]:
with open('xgb_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

In [16]:
log_predictions = loaded_model.predict(df_test)

In [17]:
predictions = np.expm1(log_predictions)

In [18]:
sub_df = pd.read_csv('sample_submission.csv')
sub_df['SalePrice'] = predictions
sub_df.to_csv('sample_sub_xgb.csv', index=False)

print("File kết quả cuối cùng đã được lưu tại: sample_sub_xgb.csv")

File kết quả cuối cùng đã được lưu tại: sample_sub_xgb.csv


## KẾT THÚC